<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_1_Models/Tutorial3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NABI-SNU/book/main/tutorials/Session_1_Models/Tutorial3.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# Tutorial 3: Generalized Linear Models

**Session 1: Models**

**Objective:** Extend linear regression to non-Gaussian data using link functions and likelihoods.

## Tutorial Objectives

Ordinary linear regression is a useful starting point, but many neuroscience measurements are not well described by Gaussian noise around a linear prediction. Spike counts are non-negative integers, binary choices are zeros and ones, and reaction times are positive.

Generalized linear models (GLMs) keep the linear predictor

$$
\eta = \mathbf{X}\boldsymbol{\beta},
$$

but pass it through a **link function** so the model prediction has the right form for the data.

By the end, you will be able to:

- Identify the three ingredients of a GLM: linear predictor, link function, and observation distribution.
- Explain why count data are often modeled with a Poisson GLM.
- Fit a Poisson regression model with gradient descent.
- Compare ordinary least squares and Poisson GLMs for spike-count data.

In [ ]:
# Imports and shared settings
import numpy as np
import matplotlib.pyplot as plt

SEED = 4
rng = np.random.default_rng(SEED)

plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
def add_intercept(x):
    """Create a two-column design matrix [1, x]."""
    x = np.asarray(x).reshape(-1)
    return np.column_stack([np.ones_like(x), x])


def poisson_rate(X, beta):
    """Compute the Poisson GLM rate with a log link."""
    eta = X @ beta
    return np.exp(np.clip(eta, -20, 20))


def poisson_nll(X, y, beta):
    """Poisson negative log likelihood, ignoring constants that do not depend on beta."""
    eta = X @ beta
    mu = np.exp(np.clip(eta, -20, 20))
    return np.mean(mu - y * eta)


def fit_poisson_glm(X, y, lr=0.05, n_steps=4000):
    """Fit a Poisson GLM with gradient descent."""
    beta = np.zeros(X.shape[1])
    losses = []

    for step in range(n_steps):
        eta = X @ beta
        mu = np.exp(np.clip(eta, -20, 20))
        grad = X.T @ (mu - y) / len(y)
        beta -= lr * grad

        if step % 50 == 0:
            losses.append(poisson_nll(X, y, beta))

    return beta, np.array(losses)

## Why Generalize Linear Regression?

In ordinary linear regression, predictions can be any real number. That is fine for many continuous measurements, but it is awkward for count data: a linear model can predict negative spike counts.

A GLM fixes this by separating two ideas:

- The **linear predictor** combines features: $\eta = \mathbf{X}\boldsymbol{\eta}$.
- The **inverse link function** maps that predictor to the expected measurement.

For a Poisson GLM with a log link, the expected count is

$$
\mu = \exp(\eta) = \exp(\mathbf{X}\boldsymbol{\eta}).
$$

Because exponentials are always positive, this model can represent firing rates or spike counts without producing negative rates.

In [ ]:
# Simulate a stimulus feature and spike counts from a Poisson GLM
n_samples = 120
x = rng.uniform(-2, 2, n_samples)
X = add_intercept(x)

beta_true = np.array([0.2, 0.9])
rate_true = poisson_rate(X, beta_true)
y = rng.poisson(rate_true)

print('First five spike counts:', y[:5])
print('Mean spike count:', y.mean().round(2))

In [ ]:
# Visualize the simulated data
x_grid = np.linspace(x.min(), x.max(), 200)
X_grid = add_intercept(x_grid)

fig, ax = plt.subplots()
ax.scatter(x, y, alpha=0.7, label='observed counts')
ax.plot(x_grid, poisson_rate(X_grid, beta_true), color='C1', linewidth=3,
        label='true expected count')
ax.set(xlabel='stimulus feature x', ylabel='spike count', title='Poisson count data')
ax.legend()
plt.show()

## Fit a Poisson GLM

For Poisson observations, the likelihood of a count $y_i$ with expected count $\mu_i$ is

$$
p(y_i \mid \mu_i) = \frac{\mu_i^{y_i} e^{-\mu_i}}{y_i!}.
$$

With the log link, $\mu_i = \exp(\mathbf{x}_i^{\top} \boldsymbol{\eta})$. We fit the model by choosing $\boldsymbol{\eta}$ that minimizes the negative log likelihood.

In [ ]:
beta_hat, losses = fit_poisson_glm(X, y)

print('True beta:     ', beta_true.round(3))
print('Estimated beta:', beta_hat.round(3))
print('Final NLL:     ', losses[-1].round(3))

fig, ax = plt.subplots()
ax.plot(np.arange(len(losses)) * 50, losses)
ax.set(xlabel='gradient step', ylabel='negative log likelihood', title='Poisson GLM training')
plt.show()

In [ ]:
# Plot the fitted expected count curve
rate_hat = poisson_rate(X_grid, beta_hat)

fig, ax = plt.subplots()
ax.scatter(x, y, alpha=0.7, label='observed counts')
ax.plot(x_grid, poisson_rate(X_grid, beta_true), color='C1', linewidth=3,
        label='true expected count')
ax.plot(x_grid, rate_hat, color='C3', linewidth=3, linestyle='--',
        label='fitted Poisson GLM')
ax.set(xlabel='stimulus feature x', ylabel='spike count', title='Fitted Poisson GLM')
ax.legend()
plt.show()

## Exercise: Implement the Poisson GLM Gradient

The gradient of the average Poisson negative log likelihood is

$$
\nabla_{\boldsymbol{\eta}} \mathcal{L} = \frac{1}{N} \mathbf{X}^{\top} (\boldsymbol{\mu} - \mathbf{y}),
$$

where $\boldsymbol{\mu} = \exp(\mathbf{X}\boldsymbol{\eta})$.

Complete `poisson_gradient` below. It should return one gradient value per model parameter.

In [ ]:
def poisson_gradient(X, y, beta):
    """Gradient of the average Poisson negative log likelihood."""
    ########################################################################
    ## TODO for students: compute the Poisson GLM gradient
    # Fill out function and remove
    raise NotImplementedError("Student exercise: compute the Poisson gradient")
    ########################################################################

    eta = ...
    mu = ...
    grad = ...

    return grad


test_grad = poisson_gradient(X, y, beta_hat)
print('Gradient near fitted beta:', test_grad.round(4))

## Compare to Ordinary Linear Regression

Ordinary least squares can still fit a line to count data, but the identity-link prediction is not constrained to be positive. This is a modeling mismatch, especially when counts are small or rates change multiplicatively.

In [ ]:
beta_ols = np.linalg.pinv(X) @ y
y_ols_grid = X_grid @ beta_ols

print('Minimum OLS prediction on grid:', y_ols_grid.min().round(2))
print('Minimum Poisson prediction on grid:', rate_hat.min().round(2))

fig, ax = plt.subplots()
ax.scatter(x, y, alpha=0.7, label='observed counts')
ax.plot(x_grid, y_ols_grid, color='C0', linewidth=3, label='OLS prediction')
ax.plot(x_grid, rate_hat, color='C3', linewidth=3, label='Poisson GLM rate')
ax.axhline(0, color='k', linewidth=1, linestyle=':')
ax.set(xlabel='stimulus feature x', ylabel='predicted count', title='Identity link vs log link')
ax.legend()
plt.show()